# train_models_v2: rebuild the feature contract through the shared pipeline

**What this does:** builds `feature_meta.json` schema 2 (string labels) from the **training partition only**, using `Python/src/preprocessing.py`. It then proves that this reproduces the exact matrix the existing models were fitted on.

**What this does not do:** it doesn't change, refit or tune any model unless `OVERWRITE_MODELS = True`. Hyperparameters are copied verbatim from `train_models.ipynb` (v1). This notebook never loads the test set.

In [ ]:
import sys, json, pickle, hashlib, platform
from pathlib import Path
import numpy as np, pandas as pd, sklearn

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    while True:
        if (cur / "Python").is_dir() and (cur / "RustSniffer").is_dir():
            return cur
        if cur.parent == cur:
            raise RuntimeError("Could not find NIDS root (folder containing Python/ and RustSniffer/).")
        cur = cur.parent

NB_DIR       = Path.cwd()                                   # Python/Notebooks/src
PROJECT_ROOT = find_project_root(NB_DIR)
SRC_DIR      = PROJECT_ROOT / "Python" / "src"
MODEL_DIR    = SRC_DIR / "models"                           # the ONLY artefact dir v2 uses (detector.py loads from here)
DATA_DIR     = NB_DIR / "datasets"
TRAIN_CSV    = DATA_DIR / "UNSW_NB15_training_set.csv"
TEST_CSV     = DATA_DIR / "UNSW_NB15_testing_set.csv"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from preprocessing import FeaturePipeline, FEATURE_COLS, CAT_COLS, SchemaError

def sha12(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()[:12] if p.exists() else "MISSING"

print("python", platform.python_version(), "| sklearn", sklearn.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_DIR   :", MODEL_DIR)
for f in ["scaler.pkl", "isolation_forest.pkl", "local_outlier_factor.pkl", "one_class_svm.pkl", "feature_meta.json"]:
    print(f"  {f:26s} {sha12(MODEL_DIR / f)}")
OLD_DIR = PROJECT_ROOT / "Python" / "Notebooks" / "Python" / "models"   # where v1 cell 8 / evaluations.ipynb also read and wrote
if OLD_DIR.exists():
    print("Second model dir exists (v1 leftover):", OLD_DIR)
    for f in ["scaler.pkl", "isolation_forest.pkl", "local_outlier_factor.pkl", "one_class_svm.pkl"]:
        print(f"  {f:26s} {sha12(OLD_DIR / f)}")

OVERWRITE_MODELS = False   # leave False: the existing pickles are kept if the check below passes

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
pipe = FeaturePipeline.fit_vocab(df_train)
X_raw, unknown = pipe.encode_frame(df_train)
y_train = df_train["label"].astype(int).to_numpy()
normal = y_train == 0
X_normal = X_raw[normal]

print("train", df_train.shape, "| normal rows", int(normal.sum()), "| unknown labels", unknown)
for c in CAT_COLS:
    print(f"{c}: {len(pipe.vocab[c])} labels | first 8: {pipe.vocab[c][:8]}")
print("state labels:", pipe.vocab["state"])

## Check: does the shared pipeline reproduce what the existing models were trained on?
LOF (`novelty=True`) stores its scaled training matrix. If our encoding, column order and scaler reproduce it exactly, row for row, then the existing models were trained on exactly this preprocessing. In that case the new contract can be written without retraining.

In [ ]:
with open(MODEL_DIR / "scaler.pkl", "rb") as f: scaler_old = pickle.load(f)
with open(MODEL_DIR / "local_outlier_factor.pkl", "rb") as f: lof_old = pickle.load(f)

checks = {
    "scaler n_samples_seen_ == normal rows": int(scaler_old.n_samples_seen_) == len(X_normal),
    "scaler mean_ == mean of rebuilt normals": np.allclose(scaler_old.mean_, X_normal.mean(0), rtol=0, atol=1e-6 * (1 + np.abs(scaler_old.mean_))),
    "scaler.transform(rebuilt) == LOF _fit_X (row order)": (lof_old._fit_X.shape == X_normal.shape
        and np.allclose(scaler_old.transform(X_normal), lof_old._fit_X, rtol=0, atol=1e-9)),
}
for k, v in checks.items(): print(("PASS  " if v else "FAIL  ") + k)
ALL_PASS = all(checks.values())
if not ALL_PASS:
    d = np.abs(scaler_old.transform(X_normal) - lof_old._fit_X).max(0) if lof_old._fit_X.shape == X_normal.shape else None
    if d is not None:
        print("columns with max |diff| > 1e-9:", {c: float(v) for c, v in zip(FEATURE_COLS, d) if v > 1e-9})

In [ ]:
meta_path = MODEL_DIR / "feature_meta.json"
if ALL_PASS:
    backup = MODEL_DIR / "feature_meta.v1.json"
    if meta_path.exists() and not backup.exists():
        meta_path.replace(backup)
        print("v1 contract backed up to", backup.name)
    pipe.save_meta(meta_path)
    print("wrote", meta_path, "| sha", sha12(meta_path))
    # round-trip: the detector's loader must accept it and give identical codes
    pipe_rt = FeaturePipeline.load(MODEL_DIR)
    tr_sample = df_train.head(5000)
    print("round-trip load OK; identical encoding:",
          np.array_equal(pipe_rt.encode_frame(tr_sample)[0], pipe.encode_frame(tr_sample)[0]))
else:
    print("NOT writing feature_meta.json: the contract would not match the trained models. Paste this output.")

## Optional: refit with identical v1 hyperparameters (off by default)
Only needed if the check above fails, or for a clean v2 retrain. No hyperparameter here differs from v1.

In [ ]:
if OVERWRITE_MODELS:
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import IsolationForest
    from sklearn.neighbors import LocalOutlierFactor
    from sklearn.svm import OneClassSVM
    scaler = StandardScaler().fit(X_normal)
    Xn = scaler.transform(X_normal)
    iforest = IsolationForest(n_estimators=100, contamination=0.1, random_state=42).fit(Xn)
    lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1, novelty=True).fit(Xn)
    ocsvm = OneClassSVM(nu=0.1, kernel="rbf", gamma="auto").fit(Xn)
    for name, obj in {"scaler.pkl": scaler, "isolation_forest.pkl": iforest,
                      "local_outlier_factor.pkl": lof, "one_class_svm.pkl": ocsvm}.items():
        with open(MODEL_DIR / name, "wb") as f: pickle.dump(obj, f)
    FeaturePipeline(pipe.feature_cols, pipe.cat_cols, pipe.vocab, scaler).save_meta(MODEL_DIR / "feature_meta.json")
    print("models refit and saved to", MODEL_DIR)
else:
    print("OVERWRITE_MODELS is False: existing models untouched.")